In [75]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

In [76]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")

In [77]:
df = df.fillna(0)

In [78]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [92]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [93]:
y_train = df_train.pop("fuel_efficiency_mpg")
y_val = df_val.pop("fuel_efficiency_mpg")

In [81]:
train_dict = df_train.to_dict(orient="records")
val_dict = df_val.to_dict(orient="records")
dv = DictVectorizer(sparse=True)
df_train = dv.fit_transform(train_dict)
df_val = dv.transform(val_dict)

In [82]:
dt = DecisionTreeRegressor(max_depth=1)
model = dt.fit(df_train, y_train)

In [83]:
export_text(dt,
    feature_names=list(dv.get_feature_names_out())
)

'|--- model_year <= 1997.50\n|   |--- value: [28.61]\n|--- model_year >  1997.50\n|   |--- value: [31.17]\n'

In [84]:
from sklearn.ensemble import RandomForestRegressor

In [94]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs = -1)
rf.fit(df_train, y_train)

ValueError: could not convert string to float: 'USA'

In [86]:
from sklearn.metrics import root_mean_squared_error

In [87]:
y_pred = rf.predict(df_val)

In [88]:
rmse = root_mean_squared_error(y_val, y_pred)
round(rmse, 2)

1.83

In [95]:
n_estimator = [10, 50, 100, 150]
train_dict = df_train.to_dict(orient="records")
val_dict = df_val.to_dict(orient="records")
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [96]:
res = []
for est in n_estimator:
    rf = RandomForestRegressor(n_estimators=est, random_state=1, n_jobs = -1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    rmse = round(rmse, 3)
    res.append(rmse)
res

[1.833, 1.774, 1.77, 1.769]

In [98]:
depth = [10, 15, 20, 25]
res = []
for d in depth:
    l=[]
    for est in n_estimator:
        rf = RandomForestRegressor(max_depth=d, n_estimators=est, random_state=1, n_jobs = -1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        #rmse = round(rmse, 3)
        l.append(rmse)
    res.append(l)
res

[[1.7823475646138716,
  1.747519928302535,
  1.7439861944908972,
  1.7437162901130996],
 [1.8313896883458138,
  1.7708830497181607,
  1.7678723617281948,
  1.7667522783622134],
 [1.8373138746453748,
  1.7723767866064455,
  1.7689177484873317,
  1.768476341618912],
 [1.8368517467554855, 1.7721232000066, 1.76831988647384, 1.767927141500425]]

In [100]:
score = pd.DataFrame(res, index = depth, columns=n_estimator)
score.index.name = "max_depth"
score.columns.name = "n_estimator"

In [104]:
score.stack().idxmin()


(np.int64(10), np.int64(150))

In [105]:
rf = RandomForestRegressor(max_depth=20, n_estimators=10, random_state=1, n_jobs = -1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

In [107]:
rf.feature_importances_

array([0.05499763, 0.00795063, 0.02006223, 0.00563848, 0.06128144,
       0.01657053, 0.16071245, 0.01627193, 0.07827125, 0.29610715,
       0.00731824, 0.01940147, 0.00651495, 0.00343644, 0.04215174,
       0.20331344])

In [113]:
importances = pd.DataFrame({
    "variable": dv.get_feature_names_out(),
    "importance": rf.feature_importances_
})

In [115]:
importances.sort_values("importance", ascending=False)

,variable,importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998
14,origin=USA,0.042152
2,drivetrain=Front-wheel drive,0.020062
11,num_doors,0.019401
5,fuel_type=Diesel,0.016571


In [116]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 28.9 MB/s  0:00:02 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 26.4 MB/s  0:00:10 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [117]:
import xgboost as xgb

In [122]:
features = list (dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names= features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names= features)
watchlist = [(dtrain, "train"), (dval, "val")]

In [124]:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

In [125]:
model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist
)

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

In [ ]:
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_)